# 02 - Method A: Markov Regime-Switching Regression

**Inertia v2 - Factor Regimes** - Sprint 2

Hamilton-style 2-state Markov regime-switching model fit independently per factor.
Each model has switching mean and switching variance, fit by maximum likelihood.
We label the state with higher in-sample mean as *favorable*, then run a forward
filter through OOS observations to obtain $P(\text{favorable}_t \mid \text{obs}_{1..t})$.

Configuration:
- 5 FF5 factors fit independently
- OOS start: 1990-01
- Refit cadence: annual (12 months) on expanding window
- Min training months: 120 (10 years)
- Weight rule: linear, $w = 2p - 1$, clipped to $[-1, +1]$

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.data import build_factor_panel, FF5_FACTORS
from lib.methods import fit_predict_markov_oos
from lib.backtest import prob_to_weight, apply_weights, static_factor_returns
from lib.evaluation import perf_stats, sharpe_bootstrap_ci
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1990-01-01'

## 1. Load panel and run Method A on all five factors

In [2]:
panel = build_factor_panel(include_macro=False)

probs = {}
for f in FF5_FACTORS:
    print(f'Fitting Markov for {f}...')
    probs[f] = fit_predict_markov_oos(panel[f], oos_start=OOS_START,
                                       refit_months=12, min_train=120)
P = pd.DataFrame(probs)
print(f'\nOOS shape: {P.shape}, range {P.index.min().date()} -> {P.index.max().date()}')
P.describe().T[['mean','std','min','max']]

Fitting Markov for MKT_RF...


Fitting Markov for SMB...


Fitting Markov for HML...


Fitting Markov for RMW...


Fitting Markov for CMA...



OOS shape: (434, 5), range 1990-01-31 -> 2026-02-28


,mean,std,min,max
MKT_RF,0.5365,0.3329,0.0000,0.9860
SMB,0.5125,0.3176,0.0574,1.0000
HML,0.4795,0.3481,0.0009,1.0000
RMW,0.3636,0.3953,0.0000,1.0000
CMA,0.3921,0.3727,0.0345,1.0000


## 2. Regime probability time series (small multiples)

Five panels, one per factor. Y axis is $P(\text{favorable}_t)$ in $[0, 1]$.
Shaded regions where probability is below 0.30 highlight stressed regimes.

In [3]:
fig, axes = plt.subplots(5, 1, figsize=(FULL_COL, 7.0), sharex=True)

for ax, f in zip(axes, FF5_FACTORS):
    s = P[f]
    ax.plot(s.index, s.values, color=FACTOR_PALETTE[f], linewidth=0.9)
    ax.axhline(0.5, color=C['muted'], linewidth=0.4, linestyle='--')
    # Shade where stressed
    stressed = (s < 0.30)
    ax.fill_between(s.index, 0, 1, where=stressed, color=C['red'],
                    alpha=0.10, linewidth=0)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(f, color=FACTOR_PALETTE[f], fontsize=10, fontweight='bold')
    ax.set_yticks([0, 0.5, 1])

axes[0].set_title('Method A: filtered P(favorable regime), 1990 to 2026',
                  loc='left', color=C['ink'])
yearly_xticks(axes[-1], every=5)
save_fig(fig, '05_method_a_probs_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/05_method_a_probs_timeseries.png


## 3. Build factor-timing strategies and compare to static

In [4]:
rows = {}
timed_returns = {}
for f in FF5_FACTORS:
    p = P[f].dropna()
    w = prob_to_weight(p, mode='linear')
    timed = apply_weights(w, panel[f], cost_bps_oneway=5.0)
    static = static_factor_returns(panel[f], timed.index)
    static_aligned = static.reindex(timed.index)
    timed_returns[f] = timed['r_net']
    s_static = perf_stats(static_aligned)
    s_timed  = perf_stats(timed['r_net'])
    rows[f] = {
        'sharpe_static':  s_static.get('sharpe_ann', np.nan),
        'sharpe_timed':   s_timed.get('sharpe_ann', np.nan),
        'mean_static':    s_static.get('mean_ann', np.nan),
        'mean_timed':     s_timed.get('mean_ann', np.nan),
        'vol_static':     s_static.get('vol_ann', np.nan),
        'vol_timed':      s_timed.get('vol_ann', np.nan),
        'mdd_static':     s_static.get('max_drawdown', np.nan),
        'mdd_timed':      s_timed.get('max_drawdown', np.nan),
    }
compare = pd.DataFrame(rows).T
compare['sharpe_uplift'] = compare['sharpe_timed'] - compare['sharpe_static']
save_table(compare, '03_method_a_per_factor_compare', out_dir=TABLE_DIR)
compare

  saved: ../tables/03_method_a_per_factor_compare.{csv,md}


,sharpe_static,sharpe_timed,mean_static,mean_timed,vol_static,vol_timed,mdd_static,mdd_timed,sharpe_uplift
MKT_RF,0.5993,-0.1241,0.0906,-0.0136,0.1512,0.1094,-0.5411,-0.7299,-0.7234
SMB,0.0908,0.0919,0.0096,0.0069,0.1059,0.0749,-0.4210,-0.2580,0.0011
HML,0.1661,0.1239,0.0186,0.0105,0.1121,0.0849,-0.5779,-0.3994,-0.0422
RMW,0.4646,0.1243,0.0418,0.0103,0.0900,0.0830,-0.4178,-0.5336,-0.3403
CMA,0.2705,0.2682,0.0204,0.0167,0.0755,0.0623,-0.2725,-0.1940,-0.0023


## 4. Per-factor Sharpe bar chart (timed vs static)

In [5]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.6))
x = np.arange(len(FF5_FACTORS))
w = 0.36

bars_s = ax.bar(x - w/2, compare['sharpe_static'].values, w,
                color=C['blue'], label='Static factor', edgecolor='white', linewidth=0.5)
bars_t = ax.bar(x + w/2, compare['sharpe_timed'].values,  w,
                color=C['purple'], label='Method A timed', edgecolor='white', linewidth=0.5)

ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(FF5_FACTORS, fontsize=10)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Method A: timed vs static Sharpe, by factor (1990 to 2026)',
             loc='left', color=C['ink'])

# Value labels above each bar (no overlap with bars)
ax.set_ylim(min(compare[['sharpe_static','sharpe_timed']].min().min(), 0) - 0.15,
            compare[['sharpe_static','sharpe_timed']].max().max() + 0.20)
bar_value_labels(ax, bars_s, fmt='{:+.2f}', offset=0.03,
                 fontsize=8, color=C['blue'])
bar_value_labels(ax, bars_t, fmt='{:+.2f}', offset=0.03,
                 fontsize=8, color=C['purple'])

legend_below(ax, ncol=2)
save_fig(fig, '06_method_a_sharpe_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/06_method_a_sharpe_bars.png


## 5. Method A composite portfolio (equal-weight across 5 timed factors)

If you treat each timed factor as a sleeve and take equal weight, you get
a composite Method A timing portfolio. We benchmark this against an equal-weight
static FF5 portfolio over the same window.

In [6]:
tr_df = pd.DataFrame(timed_returns).dropna(how='all')
method_a_composite = tr_df.mean(axis=1)  # equal weight across 5 timed sleeves

static_df = pd.DataFrame({f: panel[f].shift(-1).reindex(tr_df.index) for f in FF5_FACTORS}).dropna(how='all')
static_composite = static_df.mean(axis=1)

stats = pd.DataFrame({
    'static_eq_weight': perf_stats(static_composite),
    'method_a_timed':   perf_stats(method_a_composite),
}).T
save_table(stats, '04_method_a_composite_perf', out_dir=TABLE_DIR)
stats

  saved: ../tables/04_method_a_composite_perf.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
static_eq_weight,433.0000,0.0362,0.0491,0.7375,-0.1871,2.6831,-0.1508
method_a_timed,433.0000,0.0062,0.0393,0.1571,0.3863,4.2265,-0.2273


In [7]:
# Cumulative return chart: composite static vs composite Method A
fig, ax = plt.subplots(figsize=(FULL_COL, 3.4))
for r, color, label, lw in [
    (static_composite, C['blue'],   'Equal-weight static FF5', 1.0),
    (method_a_composite, C['purple'], 'Method A composite (timed)', 1.4),
]:
    cum = (1 + r).cumprod()
    ax.plot(cum.index, cum.values, color=color, linewidth=lw, label=label)

ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Method A composite vs static FF5, 1990 to 2026',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=5)
legend_below(ax, ncol=2)
save_fig(fig, '07_method_a_composite_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/07_method_a_composite_cumret.png


## 6. Save the OOS probability matrix

Persist the per-factor regime probabilities so Notebook 05 (ensemble) can
consume Method A's output without re-running the slow Markov fits.

In [8]:
P.to_csv(f'{TABLE_DIR}/05_method_a_probs.csv')
tr_df.to_csv(f'{TABLE_DIR}/06_method_a_timed_returns.csv')
print(f'Saved: {TABLE_DIR}/05_method_a_probs.csv  shape={P.shape}')
print(f'Saved: {TABLE_DIR}/06_method_a_timed_returns.csv  shape={tr_df.shape}')

Saved: ../tables/05_method_a_probs.csv  shape=(434, 5)
Saved: ../tables/06_method_a_timed_returns.csv  shape=(433, 5)


## Takeaways

- The Markov regime-switching model produces strongly bimodal probabilities. The model is usually quite confident which regime is active.
- Sharpe uplifts vary across factors. Some factors gain meaningfully from regime timing, others lose.
- The composite (equal-weight across all five timed sleeves) gives an apples-to-apples comparison vs static FF5 equal-weight.

**Saved artifacts**:

Figures: `05_method_a_probs_timeseries`, `06_method_a_sharpe_bars`, `07_method_a_composite_cumret`

Tables: `03_method_a_per_factor_compare`, `04_method_a_composite_perf`, `05_method_a_probs`, `06_method_a_timed_returns`